# base

In [ ]:
///- --test static

open testing

## base

### emit

In [ ]:
inl emit forall t. (x : t) : t =
    $'!x '

### emit_unit

In [ ]:
inl emit_unit forall t. (x : t) : () =
    $'!x '

### use

In [ ]:
inl use forall t. (x : t) : t =
    backend_switch {
        Gleam = fun () => $'use !x = !x ' : ()
        Lua = fun () => $'use !x = !x ' : ()
        Fsharp = fun () => $'use !x = !x ' : ()
        Python = fun () => $'use !x = !x ' : ()
        Cpp = fun () => $'use !x = !x ' : ()
        TypeScript = fun () => $'use !x = !x ' : ()
        Rust = fun () =>
            global """
                struct SpiralUse(std::rc::Rc<dyn Fn()>);
                impl Drop for SpiralUse {
                    fn drop(&mut self) { (self.0)() }
                }
                """
            $'let _spiral_use = SpiralUse((!x).clone())' : ()
    }
    $'!x '

### unit

In [ ]:
nominal unit = $'unit'

#### run_target (test)

In [1]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

run_target function
    | Gleam => fun () => $'1 + 1'
    | Lua => fun () => $'1 + 1'
    | Fsharp => fun () => $'1uy + 1uy'
    | Cuda => fun () => $'1 + 1'
    | Rust => fun () => $'1u8 + 1u8'
    | TypeScript => fun () => $'1 + 1'
    | _ => fun () => $'2uy + 2uy'
|> _assert_eq 2u8

.gleam (gleam)
__assert_eq { expected = 2 }
.lua (lua)
__assert_eq { expected = 2 }
.fsx (fsharp)
__assert_eq { expected = 2 }
.py (cuda)
__assert_eq { expected = 2 }
.rs (rust)
__assert_eq { expected = 2 }
.ts (typescript)
__assert_eq { expected = 2 }

0

### eval

In [ ]:
inl eval fn =
    fn ()

### flip

In [ ]:
inl flip fn a b =
    fn b a

### do

In [ ]:
inl do (body : () -> ()) : () =
    !!!!Do (body())

### let'

In [ ]:
inl let' fn =
    inl result : unit =
        backend_switch {
            Gleam = fun () => $'Nil' : unit
            Lua = fun () => $'nil' : unit
            Fsharp = fun () => $'()' : unit
            Python = fun () => $'None' : unit
        }
    backend_switch {
        Gleam = fun () =>
            $'let _let\'_!result = {' : ()
            fn |> indent
            $'}' : ()
        Lua = fun () =>
            $'local _get_let_!result = function()' : ()
            fn |> indent
            $'end' : ()
            $'local _let_!result = _get_let_!result()' : ()
        Fsharp = fun () =>
            $'let _let\'_!result =' : ()
            fn |> indent
        Python = fun () =>
            $'def _let\'_!result():' : ()
            fn |> indent
    }
    $'_let\'_!result '

### exec_unit

In [ ]:
inl exec_unit (fn : () -> ()) : () =
    fn ()

### lazy

In [ ]:
nominal lazy t = $'Lazy<`t>'

### memoize

In [ ]:
inl memoize forall t. (fn : () -> t) : () -> t =
    inl body () =
        inl result = mut None
        inl computed = mut false
        fun () =>
            if *computed
            then *result
            else
                result <- fn () |> Some
                computed <- true
                *result
            |> optionm.value
        : () -> t
    backend_switch {
        Gleam = fun () => body ()
        Lua = fun () => body ()
        Fsharp = fun () =>
            inl result : lazy t = $'lazy !fn ()'
            fun () => $'!result.Value' : t
        Python = fun () => body ()
        Rust = fun () => body ()
        TypeScript = fun () => body ()
        Zig = fun () => body ()
        Lean = fun () => body ()
    }

In [2]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

inl count = mut 0i32
inl add =
    fun () =>
        count <- *count + 1
        count
    |> memoize

add () |> ignore
add () |> ignore
add () |> ignore

*count
|> _assert_eq 1

.fsx (fsharp)
__assert_eq { expected = 1 }
.py (cuda)
__assert_eq { expected = 1 }
.rs (rust)
__assert_eq { expected = 1 }
.ts (typescript)
__assert_eq { expected = 1 }

0

### capture

In [ ]:
inl capture forall t. (fn : () -> t) : t =
    backend_switch {
        Gleam = fun () =>
            fn ()
        Lua = fun () =>
            fn ()
        Fsharp = fun () =>
            inl result = dyn true
            $'let mutable _capture_!result : `t option = None '
            $'('
            $'(fun () ->'
            $'(fun () ->'
            fn () |> emit_unit
            $')'
            $'|> fun x -> x ()'
            $') () )'
            $'|> fun x -> _capture_!result <- Some x'
            $'match _capture_!result with Some x -> x | None -> failwith "base.capture / _capture_!result=None"' : t
        Python = fun () =>
            fn ()
        Rust = fun () => fn ()
        Zig = fun () => fn ()
        Lean = fun () => fn ()
    }

### yield_from

In [ ]:
inl yield_from forall (t : * -> *) u. (a : t u) : () =
    backend_switch {
        Fsharp = fun () => $'yield\! !a ' : ()
        Python = fun () => $'asyncio.run(!a())' : ()
    }

### join_body

In [ ]:
inl join_body body acc x =
    if var_is x |> not
    then body acc x
    else
        inl acc = dyn acc
        join body acc x

In [ ]:
///- --test static

inl rec fold_list f s = function
    | Cons (x, x') => fold_list f (f s x) x'
    | Nil => s

In [3]:
///- --test --print-code
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

[ 5i32; 4; join 3; 2; 1 ]
|> fold_list (+) 0
|> _assert_eq 15

.gleam (gleam)
__assert_eq { expected = 15 }
.lua (lua)
__assert_eq { expected = 15 }
.fsx (fsharp)
__assert_eq { expected = 15 }
.py (cuda)
__assert_eq { expected = 15 }
.rs (rust)
__assert_eq { expected = 15 }
.ts (typescript)
__assert_eq { expected = 15 }

0

In [4]:
///- --test --print-code
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

[ 5i32; 4; join 3; 2; 1 ]
|> fold_list (join_body (+)) 0
|> _assert_eq 15

.gleam (gleam)
__assert_eq { expected = 15 }
.fsx (fsharp)
__assert_eq { expected = 15 }
.py (cuda)
__assert_eq { expected = 15 }
.rs (rust)
__assert_eq { expected = 15 }
.ts (typescript)
__assert_eq { expected = 15 }

0

### join_body_unit

In [ ]:
inl join_body_unit body d x =
    if var_is d |> not
    then body x
    else
        inl x = dyn x
        join body x

In [5]:
///- --test --print-code
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

[ 5i32; 4; join 3; 2; 1 ]
|> fold_list (fun acc n => join_body_unit ((+) acc) n n) 0
|> _assert_eq 15

.gleam (gleam)
__assert_eq { expected = 15 }
.fsx (fsharp)
__assert_eq { expected = 15 }
.py (cuda)
__assert_eq { expected = 15 }
.rs (rust)
__assert_eq { expected = 15 }
.ts (typescript)
__assert_eq { expected = 15 }

0

### (+.)

In [ ]:
inl (+.) forall t. (a : t) (b : t) : t =
    !!!!Add(a,b)

In [6]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

($'3' : i32) +. ($'-6' : i32)
|> _assert_eq -3i32

.gleam (gleam)
__assert_eq { expected = -3 }
.fsx (fsharp)
__assert_eq { expected = -3 }
.py (cuda)
__assert_eq { expected = -3 }
.rs (rust)
__assert_eq { expected = -3 }
.ts (typescript)
__assert_eq { expected = -3 }

0

### (-.)

In [ ]:
inl (-.) forall t. (a : t) (b : t) : t =
    !!!!Sub(a,b)

In [7]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

($'3' : i32) -. ($'6' : i32)
|> _assert_eq -3i32

.gleam (gleam)
__assert_eq { expected = -3 }
.fsx (fsharp)
__assert_eq { expected = -3 }
.py (cuda)
__assert_eq { expected = -3 }
.rs (rust)
__assert_eq { expected = -3 }
.ts (typescript)
__assert_eq { expected = -3 }

0

### (*.)

In [ ]:
inl (*.) forall t. (a : t) (b : t) : t =
    !!!!Mult(a,b)

In [8]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

($'3' : i32) *. ($'-1' : i32)
|> _assert_eq -3i32

.gleam (gleam)
__assert_eq { expected = -3 }
.fsx (fsharp)
__assert_eq { expected = -3 }
.py (cuda)
__assert_eq { expected = -3 }
.rs (rust)
__assert_eq { expected = -3 }
.ts (typescript)
__assert_eq { expected = -3 }

0

### (/.)

In [ ]:
inl (/.) forall t. (a : t) (b : t) : t =
    !!!!Div(a,b)

In [9]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

($'-3' : i32) /. ($'1' : i32)
|> _assert_eq -3i32

.gleam (gleam)
__assert_eq { expected = -3 }
.fsx (fsharp)
__assert_eq { expected = -3 }
.py (cuda)
__assert_eq { expected = -3 }
.rs (rust)
__assert_eq { expected = -3 }
.ts (typescript)
__assert_eq { expected = -3 }

0

### (=.)

In [ ]:
inl (=.) forall t. (a : t) (b : t) : bool =
    backend_switch {
        Gleam = fun () => $'!a == !b ' : bool
        Lua = fun () =>
            if (real typecase t with i64 => true | u64 => true | _ => false)
            then $'spiral_w64.eq(!a, !b)' : bool
            else $'!a == !b ' : bool
        Fsharp = fun () => $'!a = !b ' : bool
        Python = fun () => $'!a == !b ' : bool
        Rust = fun () => $'!a == !b ' : bool
        TypeScript = fun () => $'!a === !b ' : bool
        Zig = fun () =>
            if (real typecase t with string => true | _ => false)
            then $'std.mem.eql(u8, !a, !b)' : bool
            else $'(!a == !b)' : bool
        Lean = fun () => $'(!a == !b)' : bool
    }

In [10]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

(($'-3' : i32) =. ($'-3' : i32))
|> _assert_eq true

.gleam (gleam)
__assert_eq { expected = true }
.fsx (fsharp)
__assert_eq { expected = true }
.py (cuda)
__assert_eq { expected = true }
.rs (rust)
__assert_eq { expected = true }
.ts (typescript)
__assert_eq { expected = true }

0

### (<>.)

In [ ]:
inl (<>.) forall t. (a : t) (b : t) : bool =
    backend_switch {
        Gleam = fun () => $'!a \!= !b ' : bool
        Lua = fun () =>
            if (real typecase t with i64 => true | u64 => true | _ => false)
            then $'(not spiral_w64.eq(!a, !b))' : bool
            else $'!a ~= !b ' : bool
        Fsharp = fun () => $'!a <> !b ' : bool
        Python = fun () => $'!a \!= !b ' : bool
        Rust = fun () => $'!a \!= !b ' : bool
        TypeScript = fun () => $'!a \!== !b ' : bool
        Zig = fun () =>
            if (real typecase t with string => true | _ => false)
            then $'(\!std.mem.eql(u8, !a, !b))' : bool
            else $'(!a \!= !b)' : bool
        Lean = fun () => $'(!a \!= !b)' : bool
    }

In [11]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

(($'-3' : i32) <>. ($'3' : i32))
|> _assert_eq true

.gleam (gleam)
__assert_eq { expected = true }
.fsx (fsharp)
__assert_eq { expected = true }
.py (cuda)
__assert_eq { expected = true }
.rs (rust)
__assert_eq { expected = true }
.ts (typescript)
__assert_eq { expected = true }

0

### (<>..)

In [ ]:
inl (<>..) a b =
    fun () => a = b
    |> dyn
    |> eval
    |> not

In [12]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

(($'-3' : i32) <>.. ($'3' : i32))
|> _assert_eq true

.gleam (gleam)
__assert_eq { expected = true }
.fsx (fsharp)
__assert_eq { expected = true }
.py (cuda)
__assert_eq { expected = true }
.rs (rust)
__assert_eq { expected = true }
.ts (typescript)
__assert_eq { expected = true }

0

### append

In [ ]:
prototype append t : t -> t -> t

### (++)

In [ ]:
inl (++) a b =
    b |> append a

### pair

In [ ]:
type pair_switch a b =
    {
        Gleam : $'\#(`a, `b  )'
        Lua : $'(`a, `b)'
        Fsharp : $'(`a * `b)'
        Python : $'(`a, `b)'
        Rust : $'(`a, `b)'
        TypeScript : $'[`a, `b]'
        Zig : $'\@Tuple(&.{ `a, `b })'
    }
nominal pair a b = $'backend_switch `(pair_switch a b)'

inl pair x y =
    x, y

In [13]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

pair 1i32 2i32
|> _assert_eq (1, 2)

.gleam (gleam)
__assert_eq { expected = (1, 2) }
.fsx (fsharp)
__assert_eq { expected = (1, 2) }
.py (cuda)
__assert_eq { expected = (1, 2) }
.rs (rust)
__assert_eq { expected = (1, 2) }
.ts (typescript)
__assert_eq { expected = (1, 2) }

0

### new_pair

In [ ]:
inl new_pair forall a b. (a : a) (b : b) : pair a b =
    backend_switch {
        Gleam = fun () =>
            $'\#(!a, !b )' : pair a b
        Lua = fun () =>
            $'{!a, !b}' : pair a b
        Fsharp = fun () =>
            $'!a, !b ' : pair a b
        Python = fun () =>
            $'!a, !b ' : pair a b
        Rust = fun () =>
            inl a, b = join (a, b)
            $'(!a, !b)' : pair a b
        TypeScript = fun () => $'[!a, !b]' : pair a b
        Zig = fun () => $'.{ !a, !b }' : pair a b
    }

### from_pair

In [ ]:
inl from_pair forall a b. (pair : pair a b) : a * b =
    backend_switch {
        Gleam = fun () => ($'!pair.0' : a), ($'!pair.1' : b)
        Lua = fun () => ($'!pair[1]' : a), ($'!pair[2]' : b)
        Fsharp = fun () => ($'fst !pair ' : a), ($'snd !pair ' : b)
        Python = fun () => ($'!pair[0]' : a), ($'!pair[1]' : b)
        Rust = fun () => ($'!pair.0.clone()' : a), ($'!pair.1.clone()' : b)
        TypeScript = fun () => ($'!pair[0]' : a), ($'!pair[1]' : b)
        Zig = fun () => ($'!pair[0]' : a), ($'!pair[1]' : b)
    }

In [14]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

new_pair "a" (new_pair 1i32 "b")
|> from_pair
|> fun a, b => a, join from_pair b
|> _assert_eq' ("a", (1i32, "b"))

.gleam (gleam)
__assert_eq' { expected = (a, 1, b) }
.fsx (fsharp)
__assert_eq' { expected = (a, 1, b) }
.py (cuda)
__assert_eq' { expected = (a, 1, b) }
.rs (rust)
__assert_eq' { expected = (a, 1, b) }
.ts (typescript)
__assert_eq' { expected = (a, 1, b) }

0

#### from_pair_twice (test)

In [15]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

inl x, y = new_pair (dyn 1i32) (dyn "a") |> from_pair
inl z, w = new_pair (dyn 2i32) (dyn "b") |> from_pair
(x, y, z, w) |> _assert_eq' (1i32, "a", 2i32, "b")

.gleam (gleam)
__assert_eq' { expected = (1, a, 2, b) }
.lua (lua)
__assert_eq' { expected = (1, a, 2, b) }
.fsx (fsharp)
__assert_eq' { expected = (1, a, 2, b) }
.py (cuda)
__assert_eq' { expected = (1, a, 2, b) }
.rs (rust)
__assert_eq' { expected = (1, a, 2, b) }
.ts (typescript)
__assert_eq' { expected = (1, a, 2, b) }

0

### (||>)

In [ ]:
inl (||>) (arg1, arg2) fn =
    arg2 |> fn arg1

### (||>)

#### from_pair_2 (test)

In [16]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

(3i32, 2i32)
||> fun a b => a - b
|> _assert_eq 1

.gleam (gleam)
__assert_eq { expected = 1 }
.fsx (fsharp)
__assert_eq { expected = 1 }
.py (cuda)
__assert_eq { expected = 1 }
.rs (rust)
__assert_eq { expected = 1 }
.ts (typescript)
__assert_eq { expected = 1 }

0

#### from_pair_3 (test)

In [17]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

(1i32, 2i32)
||> flip pair
|> _assert_eq (2, 1)

.gleam (gleam)
__assert_eq { expected = (2, 1) }
.fsx (fsharp)
__assert_eq { expected = (2, 1) }
.py (cuda)
__assert_eq { expected = (2, 1) }
.rs (rust)
__assert_eq { expected = (2, 1) }
.ts (typescript)
__assert_eq { expected = (2, 1) }

0

### infer

In [ ]:
nominal infer = $'_'

### infer'

In [ ]:
nominal infer' t = $'_'

### any

In [ ]:
nominal any = $'obj'

### null

In [ ]:
inl null forall t. () : t =
    backend_switch {
        Gleam = fun () => $'Nil' : t
        Lua = fun () => $'nil' : t
        Fsharp = fun () => $'null |> unbox<`t>' : t
        Python = fun () => $'None' : t
        TypeScript = fun () => $'null' : t
        Zig = fun () => $'std.mem.zeroes(`t)' : t
    }

### defaultof

In [ ]:
inl rec defaultof forall t. (() : ()) : t =
    inl run default =
        real
            typecase t with
            | string => ""
            | int => 0
            | char => '\0'
            | ~a * ~b => (defaultof `a ()), (defaultof `b ())
            | option ~u => None `u
            | ~t =>
                if real_core.union_type_is `t then
                    real_core.union_untag `t 0
                        (fun k => forall v. => real_core.nominal_create `t (k, ()))
                        (fun _ => failwith `t ($'"defaultof / invalid tag"' : string))
                else default ()
    backend_switch {
        Gleam = fun () => run (fun () => ($'"base.defaultof / invalid type / t: `t"') : t) : t
        Lua = fun () => run (fun () => ($'"base.defaultof / invalid type / t: `t"') : t) : t
        Fsharp = fun () => run (fun () => $'Unchecked.defaultof<`t>' : t) : t
        Python = fun () =>
            run (fun () =>
                real
                    typecase t with
                    | i8 => 0i8
                    | i16 => 0i16
                    | i64 => 0i64
                    | u8 => 0u8
                    | u16 => 0u16
                    | u32 => 0u32
                    | u64 => 0u64
                    | f32 => 0.0f32
                    | f64 => 0.0f64
                    | bool => false
                    | _ => $'"base.defaultof / invalid type / t: `t"' : t
                : t
            ) : t
        Rust = fun () => run (fun () => $'Default::default()' : t) : t
        Zig = fun () => run (fun () => $'std.mem.zeroes(`t)' : t) : t
        Lean = fun () => run (fun () => $'(default : `t)' : t) : t
    }

In [18]:
///- --test
///> gleam
///> fsharp

real defaultof `string ()
|> _assert_eq ""

real defaultof `i32 ()
|> _assert_eq 0i32

real defaultof `(option string) ()
|> _assert_eq (None : option string)

.gleam (gleam)
__assert_eq { expected =  }
__assert_eq { expected = 0 }
__assert_eq { expected = None }
.fsx (fsharp)
__assert_eq { expected =  }
__assert_eq { expected = 0 }
__assert_eq { expected = None }

### choice2'

In [ ]:
type choice2_switch a b =
    {
        Gleam : $'Choice<`a, `b>'
        Lua : $'Choice<`a, `b>'
        Fsharp : $'Choice<`a, `b>'
        Python : $'Choice<`a, `b>'
        Rust : $'Result<`a, `b>'
    }
nominal choice2' a b = $'backend_switch `(choice2_switch a b)'

### choice2_unbox

In [ ]:
inl choice2_unbox forall t1 t2. (choice : choice2' t1 t2) : choice2 t1 t2 =
    run_target_args (fun () => choice) function
        | Fsharp _ => fun choice =>
            inl c1of2 (x : t1) : _ _ t2 = C1of2 x
            inl c2of2 (x : t2) : _ t1 _ = C2of2 x
            inl c1of2 = join c1of2
            inl c2of2 = join c2of2
            $'match !choice with Choice1Of2 x -> !c1of2 x | Choice2Of2 x -> !c2of2 x'
        | Rust => fun choice =>
            inl c1of2 (x : t1) : _ _ t2 = C1of2 x
            inl c2of2 (x : t2) : _ t1 _ = C2of2 x
            inl c1of2 = join c1of2
            inl c2of2 = join c2of2
            $'match !choice { Ok(x) => (!c1of2)(x), Err(x) => (!c2of2)(x) }'
        | _ => fun _ => null ()

### ref

In [ ]:
nominal ref t = $'`t ref'

### new_ref

In [ ]:
inl new_ref forall t. (x : t) : ref t =
    $'ref !x '

### ref_value

In [ ]:
inl ref_value forall t. (x : ref t) : t =
    $'!x.Value'

### ref_set_value

In [ ]:
inl ref_set_value forall t. (value : t) (ref : ref t) : ref t =
    $'!ref.Value <- !value '
    ref

### exn

In [ ]:
type exn_switch =
    {
        Gleam : $'Nil'
        Lua : $'nil'
        Fsharp : $'exn'
        Python : $'BaseException'
        Rust : $'Rc<str>'
        TypeScript : $'any'
        Zig : $'[]const u8'
    }
nominal exn = $'backend_switch `(exn_switch)'

inl exn x =
    x |> $'`exn '

### try

In [ ]:
inl catch_unwind_rust forall t. (fn : () -> t) (ex_fn : exn -> t) : t =
    inl fn = dyn fn
    inl ex_fn = dyn ex_fn
    $'{ thread_local\! { static QUIET: std::cell::Cell<u32> = std::cell::Cell::new(0); } static HOOK: std::sync::Once = std::sync::Once::new(); HOOK.call_once(|| { let prev = std::panic::take_hook(); std::panic::set_hook(Box::new(move |info| if cfg\!(panic = "abort") || QUIET.with(|q| q.get()) == 0 { prev(info) })) }); QUIET.with(|q| q.set(q.get() + 1)); let r = std::panic::catch_unwind(std::panic::AssertUnwindSafe(|| (!fn)())); QUIET.with(|q| q.set(q.get() - 1)); match r { Ok(x) => x, Err(e) => (!ex_fn)(Rc::<str>::from(e.downcast_ref::<String>().map(|s| s.as_str()).or_else(|| e.downcast_ref::<&str>().copied()).unwrap_or("panic"))) } }' : t

inl try forall t. (fn : () -> t) (ex_fn : exn -> option t) : option t =
    inl some x : option t = Some x
    inl some = dyn some
    inl fn = dyn fn
    inl ex_fn = dyn ex_fn
    backend_switch {
        Gleam = fun () =>
            fn () |> some
        Lua = fun () =>
            fn () |> some
        Fsharp = fun () =>
            $'(try !fn () |> !some with ex -> !ex_fn ex)' : option t
        Python = fun () =>
            global """
                def spiral_try(fn, some, ex_fn):
                    try:
                        return some(fn())
                    except Exception as e:
                        return ex_fn(e)
                """
            $'spiral_try(!fn, !some, !ex_fn)' : option t
        Rust = fun () =>
            catch_unwind_rust (fun () => some (fn ())) ex_fn
        TypeScript = fun () =>
            $'(() => { try { return !some(!fn()); } catch (ex) { return !ex_fn(ex); } })()' : option t
        Zig = fun () =>
            fn () |> some
    }

In [19]:
///- --test --no-agree "exception text is backend-specific"
///> fsharp
///> cuda
///> rust
///> typescript

try
    fun () => a ;[ 0i32 ] |> am'.index 1i32 |> sm'.format
    (fun ex => $'!ex ' |> sm'.format_exception |> Some)
|> optionm.value
|> _assert_eq (run_target function
    | Gleam => fun () => null ()
    | Lua => fun () => null ()
    | Fsharp => fun () => join "System.IndexOutOfRangeException: Index was outside the bounds of the array."
    | Cuda => fun () => "index 1 is out of bounds for axis 0 with size 1"
    | Cpp => fun () => null ()
    | Rust => fun () => "index out of bounds: the len is 1 but the index is 1"
    | TypeScript => fun () => "RangeError: array index 1 out of bounds for length 1"
    | _ => fun () => null ()
)

.fsx (fsharp)
__assert_eq { expected = System.IndexOutOfRangeException: Index was outside the bounds of the array. }
.py (cuda)
__assert_eq { expected = index 1 is out of bounds for axis 0 with size 1 }
.rs (rust)
__assert_eq { expected = index out of bounds: the len is 1 but the index is 1 }
.ts (typescript)
__assert_eq { expected = RangeError: array index 1 out of bounds for length 1 }

0

In [20]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

try
    fun () => a ;[ 0i32 ] |> am'.index 0i32 |> sm'.format
    (fun ex => $'!ex ' |> sm'.format_exception |> Some)
|> optionm.value
|> _assert_eq "0"

.gleam (gleam)
__assert_eq { expected = 0 }
.fsx (fsharp)
__assert_eq { expected = 0 }
.py (cuda)
__assert_eq { expected = 0 }
.rs (rust)
__assert_eq { expected = 0 }
.ts (typescript)
__assert_eq { expected = 0 }

0

### try_unit

In [ ]:
inl try_unit_fsharp_python () : bool =
    backend_switch {
        Gleam = fun () => false
        Lua = fun () => false
        Fsharp = fun () => true
        Python = fun () => true
        TypeScript = fun () => false
        Cpp = fun () => false
        CppHost = fun () => false
        Delphi = fun () => false
        Cuda = fun () => false
        Rust = fun () => false
    }

inl try_unit forall t. (fn : () -> ()) (ex_fn : (() -> exn) -> ()) : t =
    run_target function
        | Rust => fun () =>
            catch_unwind_rust fn (fun ex => ex_fn fun () => ex)
            $'()' : t
        | Zig => fun () =>
            fn ()
            $'std.mem.zeroes(`t)' : t
        | _ => fun () =>
            if try_unit_fsharp_python ()
            then
                inl fn = dyn fn
                inl ex_fn = dyn ex_fn
                backend_switch {
                    Fsharp = fun () =>
                        $'try !fn () with ex -> !ex_fn (fun () -> ex)' : ()
                        $'()' : t
                    Python = fun () =>
                        global """
                            def _spiral_try_unit(fn, ex_fn):
                                try: fn()
                                except Exception as ex: ex_fn(lambda: ex)
                            """
                        $'_spiral_try_unit(!fn, !ex_fn)' : ()
                        $'None' : t
                }
            else
                backend_switch {
                    Gleam = fun () => $'// base.try_unit' : ()
                    Lua = fun () => $'-- base.try_unit' : ()
                    Fsharp = fun () => $'try' : ()
                    Python = fun () => $'try:' : ()
                }
                fn |> indent
                backend_switch {
                    Gleam = fun () => $'// base.try_unit' : ()
                    Lua = fun () => $'-- base.try_unit' : ()
                    Fsharp = fun () => $'with ex ->' : ()
                    Python = fun () => $'except Exception as ex:' : ()
                }
                fun () =>
                    inl ex = $'ex'
                    backend_switch {
                        Fsharp = fun () => $'()' : ()
                    }
                    inl ex () =
                        ex
                    ex_fn ex
                |> indent
                backend_switch {
                    Gleam = fun () => $'// base.try_unit' : ()
                    Lua = fun () => $'-- base.try_unit' : ()
                    Fsharp = fun () =>
                        $'|> ignore'
                        $'(* try_unit'
                        $'try_unit *)' : t
                    Python = fun () => $'None' : t
                }

### try_unit'

In [ ]:
inl try_unit' forall t. (ex_fn : (() -> exn) -> ()) (fn : () -> ()) : t =
    try_unit fn ex_fn

### try_finally

In [ ]:
inl try_finally forall t. (fn : () -> ()) (finally : () -> ()) : t =
    run_target function
        | Rust => fun () =>
            inl fn = dyn fn
            inl finally = dyn finally
            $'{ let r = std::panic::catch_unwind(std::panic::AssertUnwindSafe(|| (!fn)())); (!finally)(); if let Err(e) = r { std::panic::resume_unwind(e) } }' : ()
            $'()' : t
        | Zig => fun () =>
            fn ()
            finally ()
            $'std.mem.zeroes(`t)' : t
        | _ => fun () =>
            if try_unit_fsharp_python ()
            then
                inl fn = dyn fn
                inl finally = dyn finally
                backend_switch {
                    Fsharp = fun () =>
                        $'try !fn () finally !finally ()' : ()
                        $'()' : t
                    Python = fun () =>
                        global """
                            def _spiral_try_finally(fn, finally_fn):
                                try: fn()
                                finally: finally_fn()
                            """
                        $'_spiral_try_finally(!fn, !finally)' : ()
                        $'None' : t
                }
            else
                backend_switch {
                    Gleam = fun () => $'// base.try_finally' : ()
                    Lua = fun () => $'-- base.try_finally' : ()
                    Fsharp = fun () => $'try' : ()
                    Python = fun () => $'try:' : ()
                }
                fn |> indent
                backend_switch {
                    Gleam = fun () => $'// base.try_finally' : ()
                    Lua = fun () => $'-- base.try_finally' : ()
                    Fsharp = fun () => $'finally' : ()
                    Python = fun () => $'finally:' : ()
                }
                finally |> indent
                backend_switch {
                    Gleam = fun () => $'// base.try_finally' : ()
                    Lua = fun () => $'-- base.try_finally' : ()
                    Fsharp = fun () =>
                        $'(* try_finally'
                        $'try_finally *)'
                        ()
                    Python = fun () => $'// base.try_finally' : ()
                }

### (:>)

In [21]:
prototype (~:>) r : forall t. t -> r